# NB06 — Generation Evaluation and Metrics
**Project 1.1 · Phase F**

## Purpose
This notebook reads NB05's FASTA/manifest outputs, evaluates every model × sampler using the shared functions in `evaluation.py`, adds an EOS-termination check, and writes a CSV/JSON results table with diagnostic plots.

The evaluation is multi-dimensional: validity, termination, diversity, repertoire-like composition, novelty, and sequence-level VHH/physicochemical heuristics answer different questions. **Do not collapse them into one quality score.**

## Inputs and data discipline

- Run NB05 first; the data version must match its generation manifest and the trained checkpoints.
- `train.csv` is loaded as the reference for exact novelty and global amino-acid composition. This notebook does **not** open `val.csv` or `test.csv`; the held-out test set is reserved for the final frozen-model NLL/PPL step.
- The shared Drive `v1` is the known all-six-run merge. Do not present it as VHH-only before provenance is corrected. The default here is the single curated VHH pilot `SRR3544218`.
- Pairwise Levenshtein and 3-mer Jaccard use **1,000 reproducibly sampled pairs** (the pair seed is saved). The notebook also attempts CD-HIT-2D at 90% identity when installed; if unavailable, the result stays `N/A` rather than being mislabeled exact novelty.

In [ ]:
import os, sys, json, re, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start):
    for parent in (Path(start).resolve(), *Path(start).resolve().parents):
        if (parent / "utils.py").is_file() and (parent / "evaluation.py").is_file():
            return parent
    return None

ROOT_DIR = find_repo_root(Path.cwd())
if ROOT_DIR is None and Path("/content").exists():
    clone_target = Path("/content/Recherche_AI")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "-b", "main", "https://github.com/AhmadBelhaj4/Recherche_AI.git", str(clone_target)], check=True)
    ROOT_DIR = find_repo_root(clone_target)
if ROOT_DIR is None:
    raise FileNotFoundError("Open the notebook from the Recherche_AI repository, or clone it first.")

# Auto-mount Google Drive if running in Google Colab
if Path("/content").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        for cand in [
            Path("/content/drive/MyDrive/AI_Project_Shared"),
            Path("/content/drive/Shareddrives/AI_Project_Shared"),
            Path("/content/drive/My Drive/AI_Project_Shared")
        ]:
            if cand.exists():
                os.environ["AI_PROJECT_DIR"] = str(cand)
                print(f"✓ Google Drive connecté : {cand}")
                break
    except Exception as _err:
        print(f"Notice Drive Colab: {_err}")

AI_PROJECT_DIR_OVERRIDE = None  # Override seulement si chemin personnalisé
if AI_PROJECT_DIR_OVERRIDE:
    os.environ["AI_PROJECT_DIR"] = str(AI_PROJECT_DIR_OVERRIDE)
sys.path.insert(0, str(ROOT_DIR))
from utils import get_paths
from evaluation import (
    STANDARD_AA, analyze_vhh_sequence, compute_aa_composition, compute_novelty,
    evaluate_sequences, is_valid_sequence,
)

PATHS = get_paths()
DATA_VERSION = os.environ.get("DATA_VERSION_OVERRIDE") or os.environ.get("DATA_VERSION") or "SRR3544218"
DATA_DIR = PATHS["processed"] / DATA_VERSION
GEN_DIR = PATHS["results"] / "generated" / DATA_VERSION
EVAL_DIR = PATHS["results"] / "evaluations" / DATA_VERSION
EVAL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
N_PAIRS = 1_000
RUN_CDHIT_90 = True
CDHIT_THREADS = 2
CDHIT_MEMORY_MB = 8_000  # Adjust to your runtime's available RAM.

for p in (DATA_DIR / "manifest.json", DATA_DIR / "split_manifest.json", DATA_DIR / "train.csv", GEN_DIR / "generation_manifest.json"):
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}; run NB01/NB02 and NB05 for this DATA_VERSION first.")
with open(DATA_DIR / "manifest.json", encoding="utf-8") as f:
    DATA_MANIFEST = json.load(f)
with open(DATA_DIR / "split_manifest.json", encoding="utf-8") as f:
    SPLIT_MANIFEST = json.load(f)
with open(GEN_DIR / "generation_manifest.json", encoding="utf-8") as f:
    GENERATION_MANIFEST = json.load(f)
if DATA_MANIFEST.get("data_version") != DATA_VERSION or GENERATION_MANIFEST.get("data_version") != DATA_VERSION:
    raise ValueError("The data, split, and generation manifests must all have the same DATA_VERSION.")
if GENERATION_MANIFEST.get("quick_test_only"):
    print("WARNING: NB05 was run in QUICK_TEST mode. These results are smoke-test diagnostics, not the final 10,000-sequence evaluation.")
expected_final_n = 10_000
if int(GENERATION_MANIFEST.get("n_sequences_per_model_per_config", 0)) != expected_final_n:
    print(f"WARNING: generation manifest reports {GENERATION_MANIFEST.get('n_sequences_per_model_per_config')} samples per cell; final target is {expected_final_n}.")
MIN_LEN = int(DATA_MANIFEST["parameters"]["min_len"])
MAX_LEN = int(DATA_MANIFEST["parameters"]["max_len"])

train_df = pd.read_csv(DATA_DIR / "train.csv", dtype=str, keep_default_na=False)
if "sequence" not in train_df or len(train_df) == 0:
    raise ValueError("train.csv must contain a non-empty 'sequence' column.")
TRAIN_SEQUENCES = train_df["sequence"].tolist()
TRAIN_SET = set(TRAIN_SEQUENCES)
if len(TRAIN_SET) != len(TRAIN_SEQUENCES):
    print("WARNING: exact duplicates are present in train.csv; verify NB01/NB02.")
invalid_train = [s for s in TRAIN_SEQUENCES if not is_valid_sequence(s, MIN_LEN, MAX_LEN)]
if invalid_train:
    raise ValueError(f"Training reference contains {len(invalid_train):,} invalid sequences.")
TRAIN_COMPOSITION = compute_aa_composition(TRAIN_SEQUENCES)
SAMPLING_NAMES = list(GENERATION_MANIFEST["sampling_configs"].keys())
if DATA_VERSION == "v1":
    print("WARNING: v1 is the known all-six-run merge; VHH-only interpretation is not supported until provenance is corrected.")
print(f"Version={DATA_VERSION} | train reference n={len(TRAIN_SEQUENCES):,} | lengths={MIN_LEN}–{MAX_LEN}")
print(f"Split method: {SPLIT_MANIFEST.get('method', 'not recorded')} | pair count={N_PAIRS} | pair seed={SEED}")
print("Held-out test set opened: NO")

## Metric guide

| Metric | Definition / reading | Limitation |
|---|---|---|
| **Validity** | Fraction using only the 20 canonical uppercase amino acids and within the configured length bounds. Higher is better for basic format compliance; show numerator/denominator. | Not an antibody/VHH classifier and not evidence of folding or function. |
| **EOS termination** | Fraction whose NB05 header says `<EOS>` was emitted before the max-length cap. | Does not replace validity or length checks. |
| **Exact uniqueness** | `unique generated strings / total generated strings`. Low values can indicate exact mode collapse. | High uniqueness does not mean sequences are new relative to train. |
| **Normalized Levenshtein diversity** | Mean edit distance over 1,000 sampled pairs divided by the longer sequence length. Higher means more sequence-level difference. | More distance is not always better; it may reflect off-distribution sequences. |
| **3-mer Jaccard similarity** | Mean overlap of each pair's sets of contiguous 3-residue motifs. Lower means less shared local motif overlap **within generated outputs**. | This is not generated-vs-train overlap. |
| **AA-composition KL** | `D_KL(P_generated || P_train) = Σ P_generated(a) log(P_generated(a)/P_train(a))`; helper uses `alpha=1e-6` smoothing and renormalization. Zero means matching global frequencies. | Directional, marginal only; it ignores order, position, CDRs, fold, and function. |
| **Exact novelty** | Fraction absent as an exact string from `train.csv`; shown for all samples and valid-only samples. | Near-identical variants still count as novel. |
| **CD-HIT-2D 90% novelty** | Among valid samples, fraction without a training match at the specified 90% identity threshold. | Thresholded similarity, not evolutionary or biological novelty; `N/A` if CD-HIT was not run. |
| **Length / GRAVY / charge** | Mean length plus a length histogram; average Kyte–Doolittle hydropathy; approximate residue-count charge. | GRAVY/charge are coarse proxies, not solubility, stability, or experimental pH-dependent charge. |
| **Cys / FR2 / CDR3 heuristic** | Existing code checks raw 1-based cysteine windows and regex-based FR2/CDR3-like patterns. | Not IMGT/ANARCI numbering; cannot prove Cys23–Cys104 bonding or native structure. |

The shared helper's `canonical_cysteine_rate` is a positional sequence heuristic. The supplementary FR2/CDR3 flags below are also heuristics; the generated sequences are **not** numbered by IMGT/ANARCI here. The project's separate final test NLL/PPL is not measured in NB06.

In [ ]:
def read_fasta(path):
    records, header, chunks = [], None, []
    with open(path, encoding="utf-8") as handle:
        for raw in handle:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(chunks)))
                header, chunks = line[1:], []
            else:
                if header is None:
                    raise ValueError(f"Sequence before FASTA header in {path}")
                chunks.append(line)
    if header is not None:
        records.append((header, "".join(chunks)))
    return records


def eos_from_header(header):
    match = re.search(r"(?:^|\|)eos=([01])(?:\||$)", header)
    return None if match is None else match.group(1) == "1"


def wilson_interval(successes, total, z=1.96):
    if total <= 0:
        return None, None
    p = successes / total
    den = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / den
    half = z * np.sqrt((p * (1 - p) + z**2 / (4 * total)) / total) / den
    return float(max(0, center - half)), float(min(1, center + half))


generated = {}
for model_name, model_info in GENERATION_MANIFEST["models"].items():
    for sampler in SAMPLING_NAMES:
        sample_info = model_info["samples"].get(sampler)
        if sample_info is None:
            raise KeyError(f"Missing {sampler} metadata for {model_name}")
        fasta_path = PATHS["results"] / sample_info["fasta_path_from_results"]
        if not fasta_path.exists():
            raise FileNotFoundError(f"Generated FASTA missing: {fasta_path}")
        records = read_fasta(fasta_path)
        if len(records) != int(sample_info["n_returned"]):
            raise ValueError(f"FASTA/manifest count mismatch for {model_name}/{sampler}: {len(records)} vs {sample_info['n_returned']}")
        generated[(model_name, sampler)] = {
            "model": model_name,
            "slug": model_info["slug"],
            "run_id": model_info["run_id"],
            "sampler": sampler,
            "path": fasta_path,
            "sequences": [seq for _, seq in records],
            "eos_flags": [eos_from_header(header) for header, _ in records],
        }

print(f"Loaded {len(generated)} model × sampler batches")

## Compute metrics with the shared evaluator

For each cell, `evaluate_sequences` receives the matching training set/composition, the min/max bounds from `manifest.json`, and the explicit 1,000-pair setting. The Wilson intervals shown for validity and EOS rate describe finite generated-sample uncertainty only; they do not measure training-seed variation or biological sampling bias.

Exact novelty is reported separately from CD-HIT. A sequence with one mutation relative to train is exact-novel, but may fail the 90%-identity novelty screen.

In [ ]:
results_rows = []
for key, batch in generated.items():
    seqs = batch["sequences"]
    if not seqs:
        raise ValueError(f"Empty generated batch: {key}")
    report = evaluate_sequences(
        sequences=seqs,
        train_set=TRAIN_SET,
        ref_composition=TRAIN_COMPOSITION,
        min_len=MIN_LEN,
        max_len=MAX_LEN,
        n_pairs=N_PAIRS,
        seed=SEED,
    )
    valid_seqs = [s for s in seqs if is_valid_sequence(s, MIN_LEN, MAX_LEN)]
    valid_low, valid_high = wilson_interval(len(valid_seqs), len(seqs))
    known_eos = [flag for flag in batch["eos_flags"] if flag is not None]
    eos_n = sum(known_eos)
    eos_rate = (eos_n / len(known_eos)) if known_eos else None
    eos_low, eos_high = wilson_interval(eos_n, len(known_eos))

    # Supplementary outputs from simple regex/position rules, only on valid sequences.
    diagnostics = [analyze_vhh_sequence(s) for s in valid_seqs]
    fr2_rate = float(np.mean([d["has_fr2_hallmarks"] for d in diagnostics])) if diagnostics else None
    cdr3_rate = float(np.mean([d["cdr3_length"] > 0 for d in diagnostics])) if diagnostics else None

    results_rows.append({
        "model": batch["model"], "run_id": batch["run_id"], "sampler": batch["sampler"],
        "n_generated": len(seqs), "n_valid": len(valid_seqs),
        "validity_rate": report["validity_rate"], "validity_95ci_low": valid_low, "validity_95ci_high": valid_high,
        "n_eos_status_known": len(known_eos), "n_eos_terminated": int(eos_n),
        "eos_termination_rate": None if eos_rate is None else round(float(eos_rate), 4),
        "eos_95ci_low": eos_low, "eos_95ci_high": eos_high,
        "uniqueness_rate": report["uniqueness_rate"],
        "levenshtein_diversity": report["levenshtein_diversity"],
        "kmer_3mer_jaccard_similarity": report["kmer_jaccard_similarity"],
        "kl_aa_composition_gen_vs_train": report["kl_divergence"],
        "exact_novelty_all_rate": report["novelty_rate"],
        "exact_novelty_valid_rate": compute_novelty(valid_seqs, TRAIN_SET) if valid_seqs else None,
        "mean_length": report["mean_length"], "mean_gravy": report["mean_gravy"],
        "mean_net_charge_approx": report["mean_net_charge"],
        "raw_index_cysteine_heuristic_rate": report["canonical_cysteine_rate"],
        "fr2_motif_heuristic_rate_valid": fr2_rate, "cdr3_pattern_rate_valid": cdr3_rate,
        "n_pairs_requested": N_PAIRS, "pair_sampling_seed": SEED,
        "fasta_path": str(batch["path"]),
        "cdhit90_novel_valid_n": None, "cdhit90_novel_valid_rate": None,
    })

summary_df = pd.DataFrame(results_rows)
show_cols = [
    "model", "sampler", "n_generated", "validity_rate", "eos_termination_rate",
    "uniqueness_rate", "levenshtein_diversity", "kmer_3mer_jaccard_similarity",
    "kl_aa_composition_gen_vs_train", "exact_novelty_valid_rate", "mean_length",
]
print(summary_df[show_cols].to_string(index=False))

## Optional required supplement: CD-HIT-2D at 90%

If `cd-hit-2d` is available, the next cell compares valid generated sequences (database 2) against training sequences (database 1). It records the executable/settings, and computes per-condition novelty among valid outputs. It allows a query to be up to the full configured length range longer than a reference (`-S2 MAX_LEN-MIN_LEN`).

CD-HIT-2D reports database-2 sequences without a database-1 match at the threshold. This complements exact novelty, but is still only a sequence-similarity screen. See the [CD-HIT User Guide](https://github.com/weizhongli/cdhit/wiki/3.-User%27s-Guide). If the tool is absent or fails, the 90% metric remains `N/A`; never substitute exact novelty.

In [ ]:
cdhit_executable = shutil.which("cd-hit-2d") or shutil.which("cdhit-2d")
cdhit_info = {
    "requested": RUN_CDHIT_90, "available": bool(cdhit_executable),
    "identity_threshold": 0.90,
    "identity_definition": "CD-HIT-2D default global identity: identical aligned residues / shorter sequence",
    "query_population": "valid generated sequences only",
    "reference_population": "train.csv for DATA_VERSION",
    "status": "not_run",
}
query_entries, query_to_condition = [], {}
for key, batch in generated.items():
    for idx, seq in enumerate(batch["sequences"], start=1):
        if is_valid_sequence(seq, MIN_LEN, MAX_LEN):
            query_id = f"{batch['slug']}|{batch['sampler']}|{idx:06d}"
            query_entries.append((query_id, seq))
            query_to_condition[query_id] = key

if not RUN_CDHIT_90:
    cdhit_info["status"] = "disabled_by_configuration"
elif not cdhit_executable:
    cdhit_info["status"] = "N/A: cd-hit-2d executable not found"
    print(cdhit_info["status"])
elif not query_entries:
    cdhit_info["status"] = "N/A: no valid generated sequences"
else:
    train_fasta = EVAL_DIR / "cdhit90_train_reference.fasta"
    query_fasta = EVAL_DIR / "cdhit90_valid_generated_queries.fasta"
    novel_fasta = EVAL_DIR / "cdhit90_generated_novel.fasta"
    with open(train_fasta, "w", encoding="utf-8") as handle:
        for idx, seq in enumerate(sorted(TRAIN_SET), start=1):
            handle.write(f">train_{idx:09d}\n{seq}\n")
    with open(query_fasta, "w", encoding="utf-8") as handle:
        for query_id, seq in query_entries:
            handle.write(f">{query_id}\n{seq}\n")
    cmd = [
        cdhit_executable, "-i", str(train_fasta), "-i2", str(query_fasta), "-o", str(novel_fasta),
        "-c", "0.9", "-n", "5", "-G", "1", "-d", "0",
        "-S2", str(MAX_LEN - MIN_LEN), "-T", str(CDHIT_THREADS), "-M", str(CDHIT_MEMORY_MB),
    ]
    cdhit_info["command"] = cmd
    cdhit_info["db2_length_difference_allowance"] = MAX_LEN - MIN_LEN
    try:
        proc = subprocess.run(cmd, check=True, capture_output=True, text=True)
        if not novel_fasta.exists():
            raise FileNotFoundError(f"CD-HIT output not created: {novel_fasta}")
        novel_ids = {header.split()[0] for header, _ in read_fasta(novel_fasta)}
        cdhit_info.update({
            "status": "completed", "n_valid_queries": len(query_entries),
            "n_unmatched_at_90pct": len(novel_ids), "novel_fasta": str(novel_fasta),
            "stdout_tail": proc.stdout[-1500:],
        })
        for row in results_rows:
            condition = (row["model"], row["sampler"])
            ids = [qid for qid, qcondition in query_to_condition.items() if qcondition == condition]
            novel_n = sum(qid in novel_ids for qid in ids)
            row["cdhit90_novel_valid_n"] = int(novel_n)
            row["cdhit90_novel_valid_rate"] = float(novel_n / len(ids)) if ids else None
        print(f"CD-HIT-2D: {len(novel_ids):,}/{len(query_entries):,} valid generated sequences had no >=90% train match.")
    except (subprocess.CalledProcessError, FileNotFoundError) as exc:
        cdhit_info["status"] = f"N/A: CD-HIT-2D failed ({exc})"
        if isinstance(exc, subprocess.CalledProcessError):
            cdhit_info["stderr_tail"] = (exc.stderr or "")[-1500:]
        for row in results_rows:
            row["cdhit90_novel_valid_n"] = None
            row["cdhit90_novel_valid_rate"] = None
        print(cdhit_info["status"])

summary_df = pd.DataFrame(results_rows)
csv_path = EVAL_DIR / f"generation_metrics_{DATA_VERSION}.csv"
json_path = EVAL_DIR / f"generation_metrics_{DATA_VERSION}.json"
summary_df.to_csv(csv_path, index=False)

def json_safe(value):
    if value is None:
        return None
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        value = value.item()
    if isinstance(value, float) and (np.isnan(value) or np.isinf(value)):
        return None
    return value

payload = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "data_version": DATA_VERSION,
    "data_manifest_created_utc": DATA_MANIFEST.get("created_utc"),
    "data_manifest_git_commit": DATA_MANIFEST.get("git_commit"),
    "split_method": SPLIT_MANIFEST.get("method"),
    "train_reference_count": len(TRAIN_SEQUENCES), "test_set_used": False,
    "n_pairs": N_PAIRS, "pair_sampling_seed": SEED,
    "metrics_implementation": "evaluation.py plus NB06 EOS/Wilson/CD-HIT fields",
    "cdhit90": cdhit_info,
    "results": [{k: json_safe(v) for k, v in row.items()} for row in results_rows],
}
json_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved CSV : {csv_path}\nSaved JSON: {json_path}")
print(summary_df[show_cols + ["cdhit90_novel_valid_rate"]].to_string(index=False))

## Diagnostic plots and interpretation

The plots compare generated outputs with the **training reference**: the first displays the full length distribution; the second displays `generated amino-acid frequency − train frequency`. They are descriptive checks, not pass/fail thresholds.

- High uniqueness can coexist with high KL divergence or invalid lengths.
- Lower within-batch 3-mer Jaccard means less local motif overlap; it is not a score to minimize without limit.
- Exact novelty can exceed CD-HIT 90% novelty because close variants are not exact copies.
- Cysteine/FR2/CDR3 heuristics are not IMGT/ANARCI numbering, structure, or binding evidence.

No target-specific binding, affinity, specificity, expression, or experimental developability is measured in this phase.

In [ ]:
# Full length histograms retain short/truncated outputs so failures remain visible.
fig, ax = plt.subplots(figsize=(12, 5.5))
bins = np.arange(0, MAX_LEN + 2) - 0.5
ax.hist([len(s) for s in TRAIN_SEQUENCES], bins=bins, density=True, histtype="step", linewidth=2.4, color="black", label="Train reference")
for key, batch in generated.items():
    ax.hist([len(s) for s in batch["sequences"]], bins=bins, density=True, histtype="step", linewidth=1.1, label=f"{key[0]} · {key[1]}")
ax.axvline(MIN_LEN, color="grey", linestyle="--", linewidth=1)
ax.axvline(MAX_LEN, color="grey", linestyle="--", linewidth=1)
ax.set(xlabel="Decoded amino-acid length", ylabel="Probability density", title=f"Length distributions — {DATA_VERSION}")
ax.legend(fontsize=8, ncol=2)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
length_plot = EVAL_DIR / "length_distributions.png"
fig.savefig(length_plot, dpi=170, bbox_inches="tight")
plt.show()
print(f"Saved {length_plot}")

# Global AA residual heatmap; it complements, but is not equivalent to, KL divergence.
labels, deltas = [], []
for key, batch in generated.items():
    comp = compute_aa_composition(batch["sequences"])
    labels.append(f"{key[0]} · {key[1]}")
    deltas.append([comp[aa] - TRAIN_COMPOSITION[aa] for aa in STANDARD_AA])
mat = np.asarray(deltas, dtype=float)
lim = max(float(np.max(np.abs(mat))) if mat.size else 0.0, 1e-4)
fig, ax = plt.subplots(figsize=(13, max(4.5, 0.55 * len(labels))))
im = ax.imshow(mat, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
ax.set_xticks(np.arange(len(STANDARD_AA)))
ax.set_xticklabels(list(STANDARD_AA))
ax.set_yticks(np.arange(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Amino acid")
ax.set_title("Global amino-acid frequency shift (generated − train)")
fig.colorbar(im, ax=ax, label="Frequency difference")
fig.tight_layout()
composition_plot = EVAL_DIR / "aa_composition_residuals.png"
fig.savefig(composition_plot, dpi=170, bbox_inches="tight")
plt.show()
print(f"Saved {composition_plot}")